In [22]:
!pip install -q --upgrade torchao

In [23]:
!pip install -q transformers peft

#Loading the fine tuned model that I created earlier

In [24]:
from transformers import pipeline

repo_name = "shloksutar/fraud-detector"

finetuned_model = pipeline("text-generation", model=repo_name, device="cuda")
print("Fine-tuned model loaded!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/224 [00:00<?, ?it/s]

Fine-tuned model loaded!


#Loading the base model

In [25]:
base_model = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device="cuda"
)
print("Base model loaded!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded!


#The dataset with 50 fraud data and 50 safe/legitimate data

In [26]:
!pip install -q datasets

In [27]:
from datasets import load_dataset

dataset = load_dataset("CiferAI/Cifer-Fraud-Detection-Dataset-AF", split="train")
print(f"Dataset size: {len(dataset):,} transactions")

Dataset size: 21,000,000 transactions


In [28]:
from datasets import concatenate_datasets

# Get 50 fraud transactions
fraud_test = dataset.filter(lambda x: x['isFraud'] == 1)
fraud_test = fraud_test.shuffle(seed=99).select(range(50))

# Get 50 legitimate transactions
non_fraud_test = dataset.filter(lambda x: x['isFraud'] == 0)
non_fraud_test = non_fraud_test.shuffle(seed=99).select(range(50))

#100 new transactions that the model was not trained on

In [29]:
test_data = concatenate_datasets([fraud_test, non_fraud_test])
test_data = test_data.shuffle(seed=99)

print(f"Test set: {len(test_data)} transactions")

Test set: 100 transactions


#Creating the prompt.

In [30]:
def create_prompt(row):
    prompt = f"""Analyze this transaction for fraud risk:
- Type: {row['type']}
- Amount: ${row['amount']:,.2f}
- Sender Balance Before: ${row['oldbalanceOrg']:,.2f}
- Sender Balance After: ${row['newbalanceOrig']:,.2f}
- Recipient Balance Before: ${row['oldbalanceDest']:,.2f}
- Recipient Balance After: ${row['newbalanceDest']:,.2f}"""
    return prompt

#Getting the actual labels or correct lables

In [31]:
actual_labels = []

for row in test_data:
    if row['isFraud'] == 1:
        actual_labels.append("HIGH")
    else:
        actual_labels.append("LOW")

print(f"Prepared {len(actual_labels)} labels")

Prepared 100 labels


#Creating the Prediction function.

In [32]:
def get_prediction(model, prompt):
    messages = [{"role": "user", "content": prompt}]
    output = model(messages, max_new_tokens=10, return_full_text=False)[0]
    response = output["generated_text"].upper()
    if "HIGH" in response:
        return "HIGH"
    elif "LOW" in response:
        return "LOW"
    else:
        return "LOW"

#Fine tuned model's prediction label & Base model's prediction label.

In [33]:
finetuned_predictions = []
base_predictions = []

for i in range(len(test_data)):
    row = test_data[i]
    prompt = create_prompt(row)

    finetuned_predictions.append(get_prediction(finetuned_model, prompt))
    base_predictions.append(get_prediction(base_model, prompt))

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(test_data)}...")

print("Complete!")

Processed 10/100...
Processed 20/100...
Processed 30/100...
Processed 40/100...
Processed 50/100...
Processed 60/100...
Processed 70/100...
Processed 80/100...
Processed 90/100...
Processed 100/100...
Complete!


In [34]:
!pip install -q scikit-learn

#Comparing finetuned model and base model prediction labels with the actual ones

In [35]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Fine-tuned model
ft_accuracy  = accuracy_score(actual_labels, finetuned_predictions)
ft_precision = precision_score(actual_labels, finetuned_predictions, pos_label="HIGH")
ft_recall    = recall_score(actual_labels, finetuned_predictions, pos_label="HIGH")
ft_f1        = f1_score(actual_labels, finetuned_predictions, pos_label="HIGH")

In [36]:
# Base model
base_accuracy  = accuracy_score(actual_labels, base_predictions)
base_precision = precision_score(actual_labels, base_predictions, pos_label="HIGH")
base_recall    = recall_score(actual_labels, base_predictions, pos_label="HIGH")
base_f1        = f1_score(actual_labels, base_predictions, pos_label="HIGH")

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#Printing the scores

In [37]:
print("=" * 50)
print("MODEL COMPARISON")
print("=" * 50)
print(f"Metric       Base Model    Fine-tuned")
print("-" * 50)
print(f"Accuracy     {base_accuracy:.0%}            {ft_accuracy:.0%}")
print(f"Precision    {base_precision:.0%}            {ft_precision:.0%}")
print(f"Recall       {base_recall:.0%}            {ft_recall:.0%}")
print(f"F1 Score     {base_f1:.0%}            {ft_f1:.0%}")
print("=" * 50)

MODEL COMPARISON
Metric       Base Model    Fine-tuned
--------------------------------------------------
Accuracy     50%            55%
Precision    0%            54%
Recall       0%            72%
F1 Score     0%            62%
